In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import folium
import re
import numpy as np

df = pd.read_csv("2022.23_PhD_NA_beachCleanData.csv")
df.head()

,beach,date,team,start.time,end.time,latitude,longitude,sea_region,location,land,...,nonremoveable_rope,nonremoveable_buriedRope,nonremoveable_fishingLine,nonremoveable_buriedFishingLine,nonremoveable_container,nonremoveable_buriedContanier,nonremoveable_tyre,nonremoveable_other,nonremoveable_nonidentifyable,comments.1
0,Lunderston Bay,24/09/2022,"NA, PD",09:15,09:45,55.9310,-4.8768,Firth of Clyde,River Clyde,Mainland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Lunderston Bay,20/11/2022,"NA, PD",11:15,12:00,55.9310,-4.8768,Firth of Clyde,River Clyde,Mainland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Girvan Bay,24/09/2022,"NA, PD",12:20,12:50,55.2416,-4.8604,Firth of Clyde,Clyde,Mainland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Girvan Bay,20/11/2022,"NA, PD",15:50,16:42,55.2416,-4.8604,Firth of Clyde,Clyde,Mainland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunure,24/09/2022,"NA, PD",13:30,13:45,55.4053,-4.7615,Firth of Clyde,Clyde,Mainland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
def top_3_plastics(plastic_row, plastic_names):
    # function to get top 3 polluting plastics
    indices = np.argsort(plastic_row)[-3:][::-1]
    return [(plastic_names[i], plastic_row[i]) for i in indices if plastic_row[i] > 0]

def normalise_object(obj):
    # split camelCase → words
    tokens = re.findall(r"[a-z]+", obj.lower())
    
    # remove size descriptors
    size_words = {"sml", "med", "lrg", "small", "medium", "large"}
    tokens = [t for t in tokens if t not in size_words]
    
    return tokens

def classify_object(tokens):
    categories = set()

    for category, keywords in OBJECT_CATEGORY_RULES.items():
        if any(t in keywords for t in tokens):
            categories.add(category)

    if not categories:
        categories.add("other")

    return list(categories)

def severity_radius(s, scale=25):
    return scale * np.sqrt(s / S.max())

def split_beaches_sampled_twice(df, beach_col="beach", date_col="date"):
    """
    Splits a DataFrame into two DataFrames based on beaches
    that were sampled exactly twice.

    df_1 contains the first sampling,
    df_2 contains the second sampling.

    Beaches sampled only once are ignored.
    """

    # Ensure dates are sortable
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True)

    # Sort so "first" and "second" are well-defined
    df = df.sort_values([beach_col, date_col])

    # Keep only beaches that appear exactly twice
    counts = df[beach_col].value_counts()
    valid_beaches = counts[counts == 2].index

    df_twice = df[df[beach_col].isin(valid_beaches)]

    # Split into first and second occurrence
    df_1 = df_twice.groupby(beach_col).nth(0).reset_index()
    df_2 = df_twice.groupby(beach_col).nth(1).reset_index()

    return df_1, df_2

In [3]:
df_1, df_2 = split_beaches_sampled_twice(df)

In [4]:
beaches = list(df_1["beach"])
coords = df_1.iloc[:, 6:8].to_numpy()
plastics_1, plastics_2 = df_1.iloc[:, 37:168], df_2.iloc[:, 37:168]
plastic_types = list(plastics_1.columns)
avg_plastics = 0.5 * (plastics_1.to_numpy() + plastics_2.to_numpy())

In [5]:
avg_plastics

array([[0. , 0.5, 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [0. , 2. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1.5, 0. , ..., 0. , 0. , 1. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ]], shape=(22, 131))

In [6]:
print(coords)
print(avg_plastics.shape)

[[55.6256 -4.7497]
 [56.5184 -6.8842]
 [56.4776 -6.9771]
 [56.4826 -6.9774]
 [55.8745 -6.1168]
 [56.5442 -6.7425]
 [55.3086 -5.6737]
 [56.4921 -6.8582]
 [56.4485 -5.6593]
 [55.4053 -4.7615]
 [56.3844 -5.5226]
 [55.2416 -4.8604]
 [56.5275 -6.7837]
 [55.6445 -5.4822]
 [55.931  -4.8768]
 [55.4238 -5.731 ]
 [56.55   -6.7593]
 [57.338  -6.1095]
 [55.6844 -6.5125]
 [57.284  -6.4569]
 [56.5294 -6.228 ]
 [56.2892 -6.2188]]
(22, 131)


In [7]:
scotland_map = folium.Map(
    location=[56.5, -4.0],
    zoom_start=6,
    tiles="OpenStreetMap"
)

for i in range(len(beaches)):
    lat, lon = coords[i]
    beach_name = beaches[i]
    plastic_row = avg_plastics[i]

    top3 = top_3_plastics(plastic_row, plastic_types)

    if not top3:
        popup_text = f"<b>{beach_name}</b><br>No plastics recorded"
    else:
        popup_text = f"<b>{beach_name}</b><br>"
        for name, value in top3:
            popup_text += f"{name}: {value}<br>"

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(icon="info-sign")
    ).add_to(scotland_map)

In [8]:
display(scotland_map)

In [9]:
split_by_dots = lambda s : ["_".join(v.split("..")) if ".." in v else v for v in s]
split_by_dash = lambda s : [ v.split("_") if "_" in v else v for v in s]

temp = split_by_dash(split_by_dots(plastic_types))
plastic_names = np.array(temp)

OBJECT_CATEGORY_RULES = {
    "fishing_gear": {
        "fishingnet", "fishline", "lobster", "crab", "pot", "buoy", "oyster", "hook"
    },
    "microplastics": {
        "fragment", "nurdle", "pellet", "foam"
    },
    "consumer_packaging": {
        "bag", "bottle", "cup", "wrapper", "container", "packaging"
    },
    "hazardous": {
        "oil", "fertiliser", "cartridge", "shotgun", "paraffin"
    },
    "personal_items": {
        "cigarette", "lighter", "pen", "shoe", "toy", "comb", "sunglasses", "glove"
    },
    "industrial": {
        "crate", "jerry", "hardhat", "strapping", "fibreglass", "drum"
    },
    "sanitary_medical": {
        "condom", "tampon", "wipe", "syringe", "facemask"
    }
}

plastic_metadata = []

for row in plastic_names:
    survey, material, obj = row
    tokens = normalise_object(obj)
    categories = classify_object(tokens)

    plastic_metadata.append({
        "material": material,
        "object": obj,
        "tokens": tokens,
        "categories": categories
    })

CATEGORY_SEVERITY = {
    "microplastics": 4.0,
    "fishing_gear": 3.5,
    "hazardous": 4.5,
    "sanitary_medical": 3.0,
    "industrial": 2.5,
    "consumer_packaging": 2.0,
    "personal_items": 1.5,
    "other": 1.0
}

MATERIAL_MULTIPLIER = {
    "plastic": 1.0,
    "polystyrene": 1.2,
    "rubber": 1.1,
    "metal": 0.8,
    "glass": 0.6,
    "wood": 0.5,
    "cloth": 0.7,
    "paper": 0.6,
    "sanitary": 1.3,
    "medical": 1.4,
    "pollutants": 2.0
}

W = []

for meta in plastic_metadata:
    cat_score = np.mean([CATEGORY_SEVERITY[c] for c in meta["categories"]])
    mat_mult = MATERIAL_MULTIPLIER.get(meta["material"], 1.0)
    W.append(cat_score * mat_mult)

W = np.array(W)
W = W / W.sum()

# Formula for Severity Score :
To calculate the severity of pollution in a certain area, we need to break it down into different categories of pollution.
For this problem, we'll stick to 8 types of pollution. This way, we are able to tackle the pollution level from different angles.
The types of pollution we can consider for this project is shown below.
- microplastics        (score : 4.0)
- fishing gear         (score : 3.5)
- hazardous            (score : 4.5)
- sanitary medical     (score : 3.0)
- industrial           (score : 2.5)
- consumer packaging   (score : 2.0)
- personal items       (score : 1.5)
- other                (score : 1.0)

Let there be an array containing all the names of the plastics $P_{Names}$ (shown as `plastic_names` in the code). Each of the names in `plastic_names` seem to be strings in the format of `"X01_plastic_gloves"` or something similar. This means that we can break down each of these strings into 3 separate components. The `X01/X02/X03/count` (survey code), the `plastic/polystyrene/metal` (material), and the `fishingnetLrg/oilDrum/fragmentSml` (object found on the shore).

$$S = \frac{1}{N}W^TP_\mu$$

In [10]:
N = avg_plastics.shape[0]
S = (W @ avg_plastics.T) / N  # where S stands for severity

In [11]:
norm = colors.Normalize(
    vmin=S.min(),
    vmax=S.max()
)

cmap = cm.get_cmap("viridis")

scotland_map = folium.Map(
    location=[56.5, -4.0],
    zoom_start=6,
    tiles="CartoDB positron"
)

for i in range(len(beaches)):
    lat, lon = coords[i]
    s = S[i]

    # Viridis colour
    rgba = cmap(norm(s))
    colour = colors.rgb2hex(rgba)

    popup_html = f"""
    <b>{beaches[i]}</b><br>
    <b>Severity score:</b> {s:.3f}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=severity_radius(s),
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{beaches[i]} | Severity: {s:.2f}"
    ).add_to(scotland_map)

display(scotland_map)

C:\Users\ishir\AppData\Local\Temp\ipykernel_27556\3072907236.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis")
